# Model visualization

__TABLE OF CONTENTS__
1. [Introduction](#intro)
2. [Setup](#setup)
3. [General Visualizations (for all models)](#general)
4. [Linear Models Visualizations](#lin)
5. [Tree-Based Models Visualizations](#tree)

<a id="intro"></a>
## 1. Introduction

This notebook centralizes a set of reusable visualization utilities designed to support model assessment across the project. Most functions operate independently of model type and accept any estimator together with its feature matrix and model name (the ones that do not are clearly specified).

The goal is to provide a consistent toolkit for inspecting model behaviour, understanding feature influence, and identifying potential sources of error or instability during evaluation.
The resulting visual diagnostics complement the quantitative metrics used throughout the benchmarking and optimization stages, ensuring that model selection is guided by both numerical performance and interpretable evidence.

<a id="setup"></a>
## 2. Setup

In [1]:
# pip install shap

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance
import shap

sns.set(style="whitegrid", font_scale=1.2)

<a id="general"></a>
## General Visualizations (for all models)

In [ ]:
def plot_pred_vs_true(model, model_name):
    """
    Plots predicted vs true price values for a given model.
    """

    y_pred = model.predict(X_full_final)

    plt.figure(figsize=(6,6))
    sns.scatterplot(x=y_full, y=y_pred, alpha=0.3, s=15)
    plt.plot([y_full.min(), y_full.max()],
             [y_full.min(), y_full.max()],
             color='red', linewidth=2)

    plt.xlabel("True Price")
    plt.ylabel("Predicted Price")
    plt.title(f"{model_name}: Predicted vs True")
    plt.tight_layout()
    plt.show()


def plot_pred_vs_true_fs(model, model_name):
    """
    Plots predicted vs true price values for a given model.
    Assumes the model was trained on log1p(price).
    """
    # predict in log-space
    y_pred_log = model.predict(X_full_sel)

    # back-transform to price
    y_pred = np.expm1(y_pred_log)

    plt.figure(figsize=(6, 6))
    sns.scatterplot(x=y_full, y=y_pred, alpha=0.3, s=15)
    plt.plot(
        [y_full.min(), y_full.max()],
        [y_full.min(), y_full.max()],
        color="red",
        linewidth=2,
    )
    plt.xlabel("True Price")
    plt.ylabel("Predicted Price")
    plt.title(f"{model_name}: Predicted vs True")
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_residuals(model, model_name):
    """
    Plots residuals vs predicted price values for a given model.
    """

    y_pred = model.predict(X_full_final)
    residuals = y_full - y_pred

    plt.figure(figsize=(7,5))
    sns.scatterplot(x=y_pred, y=residuals, alpha=0.3, s=15)
    plt.axhline(0, color='red', linewidth=2)
    plt.xlabel("Predicted Price")
    plt.ylabel("Residual (True - Pred)")
    plt.title(f"{model_name}: Residuals vs Predicted")
    plt.tight_layout()
    plt.show()


def plot_residuals_fs(model, model_name):
    """
    Plots residuals vs predicted price values for a given model.
    Assumes the model was trained on log1p(price).
    """
    # Predict in log-space, then back-transform
    y_pred_log = model.predict(X_full_sel)
    y_pred = np.expm1(y_pred_log)

    # Residuals in original price scale
    residuals = y_full - y_pred

    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=y_pred, y=residuals, alpha=0.3, s=15)
    plt.axhline(0, color="red", linewidth=2)
    plt.xlabel("Predicted Price")
    plt.ylabel("Residual (True - Pred)")
    plt.title(f"{model_name}: Residuals vs Predicted")
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_residual_distribution(model, model_name):
    """
    Plots the distribution of residuals for a given model.
    """

    y_pred = model.predict(X_full_final)
    residuals = y_full - y_pred

    plt.figure(figsize=(7,5))
    sns.histplot(residuals, kde=True, bins=50, color='steelblue')
    plt.axvline(0, color='red', linewidth=2)
    plt.xlabel("Residual")
    plt.ylabel("Count")
    plt.title(f"{model_name}: Residual Distribution")
    plt.tight_layout()
    plt.show()


def plot_residual_distribution_fs(model, model_name):
    """
    Plots the distribution of residuals for a given model.
    Assumes the model was trained on log1p(price).
    """
    # Predict in log-space, then back-transform
    y_pred_log = model.predict(X_full_sel)
    y_pred = np.expm1(y_pred_log)

    # Residuals in original price scale
    residuals = y_full - y_pred

    plt.figure(figsize=(7, 5))
    sns.histplot(residuals, kde=True, bins=50, color="steelblue")
    plt.axvline(0, color="red", linewidth=2)
    plt.xlabel("Residual")
    plt.ylabel("Count")
    plt.title(f"{model_name}: Residual Distribution")
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_permutation_importance(model, model_name, n_repeats=5, top=20):
    '''
    Measures how much the model worsens when a feature is randomly shuffled.
    Permutation importance is more trustworthy than feature_importances_.
    '''

    result = permutation_importance(
        model,
        X_full_final,
        y_full,
        n_repeats=n_repeats,
        random_state=42,
        n_jobs=-1
    )

    imp_df = pd.DataFrame({
        "feature": X_full_final.columns,
        "importance": result.importances_mean
    }).sort_values("importance", ascending=False).head(top)

    plt.figure(figsize=(10, top * 0.35 + 1))
    sns.barplot(x="importance", y="feature", data=imp_df, palette="magma")
    plt.title(f"{model_name} - Permutation Importance (Top {top})")
    plt.tight_layout()
    plt.show()


def plot_permutation_importance_fs(model, model_name, n_repeats=5, top=20):
    """
    Permutation importance for log-trained linear models.
    Uses X_full_sel (the matrix used for fitting) 
    and y_full_log (because model predicts log(price)).
    """

    result = permutation_importance(
        estimator=model,
        X=X_full_sel,
        y=y_full_log,            # match model's prediction scale
        n_repeats=n_repeats,
        random_state=42,
        n_jobs=-1
    )

    imp_df = pd.DataFrame({
        "feature": X_full_sel.columns,
        "importance": result.importances_mean
    }).sort_values("importance", ascending=False).head(top)

    plt.figure(figsize=(10, top * 0.35 + 1))
    sns.barplot(x="importance", y="feature", data=imp_df, palette="magma")
    plt.title(f"{model_name} — Permutation Importance (Top {top})")
    plt.tight_layout()
    plt.show()

In [ ]:
def apply_shap(model, X, model_name="Model", sample_size=2000):
    """
    Applies SHAP to any model by automatically selecting the correct explainer.
    Produces:
        - SHAP summary bar plot (global importance)
        - SHAP beeswarm plot (feature effect distribution)
    """

    # Sample for speed
    X_sample = X.sample(min(sample_size, len(X)), random_state=42)

    # Explainer selection
    if hasattr(model, "tree_") or hasattr(model, "estimators_"):
        # Tree-based models (RF, ExtraTrees, GBM, XGBoost, LGBM, CatBoost)
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)

    elif model.__class__.__name__ in ["LinearRegression", "Ridge", "Lasso", "ElasticNet"]:
        # Linear family
        explainer = shap.LinearExplainer(model, X)
        shap_values = explainer.shap_values(X_sample)

    else:
        # Fallback for SVM, KNN, neural nets, etc.
        background = X.sample(200, random_state=42)
        explainer = shap.KernelExplainer(model.predict, background)
        shap_values = explainer.shap_values(X_sample)

    # PLOTS
    print(f"\nSHAP for {model_name} (sample size = {len(X_sample)})")

    shap.summary_plot(shap_values, X_sample, plot_type="bar", show=True)
    shap.summary_plot(shap_values, X_sample, show=True)




def apply_shap_fs(model, X, model_name="Model", sample_size=2000):
    """
    Applies SHAP to any regression model.
    Works with:
        - Tree-based models (RF, ET, GBM, XGB, LGBM, CatBoost)
        - Linear models (LR, Ridge, Lasso, ElasticNet)
        - Any black-box model (via KernelExplainer)

    X must be the same feature matrix used in model.fit() 
    (for your FE+FS models: X_full_sel).
    """

    # ---- 1. Sample for speed ----
    X_sample = X.sample(min(sample_size, len(X)), random_state=42)

    # ---- 2. Select correct explainer ----
    if hasattr(model, "tree_") or hasattr(model, "estimators_"):
        # Tree models
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)

    elif model.__class__.__name__ in ["LinearRegression", "Ridge", "Lasso", "ElasticNet"]:
        # Linear models (interpreted in log-space)
        explainer = shap.LinearExplainer(model, X)
        shap_values = explainer.shap_values(X_sample)

    else:
        # Kernel SHAP for generic models
        background = X.sample(min(200, len(X)), random_state=42)
        explainer = shap.KernelExplainer(model.predict, background)
        shap_values = explainer.shap_values(X_sample)

    # ---- 3. SHAP PLOTS ----
    print(f"\nSHAP for {model_name} (sample size = {len(X_sample)})")

    shap.summary_plot(shap_values, X_sample, plot_type="bar", show=True)
    shap.summary_plot(shap_values, X_sample, show=True)

<a id="lin"></a>
## Linear Models Visualizations

In [ ]:
def plot_coefficients(models, model_names):
    """
    Plots the distribution of coefficients across multiple linear models.
    """

    coef_df = pd.DataFrame({
        name: model.coef_
        for model, name in zip(models, model_names)
    }, index=X_full_final.columns)

    plt.figure(figsize=(12,7))
    coef_df.plot(kind="density", figsize=(12,6))
    plt.title("Distribution of Coefficients Across Linear Models")
    plt.xlabel("Coefficient Value")
    plt.grid(True)
    plt.show()

    # Barplot of absolute coefficient magnitudes 
    top_abs = coef_df.abs().mean(axis=1).sort_values(ascending=False).head(20)
    plt.figure(figsize=(10,6))
    sns.barplot(x=top_abs.values, y=top_abs.index, orient="h")
    plt.title("Most Influential Features (Average Across Models)")
    plt.xlabel("Average |coefficient|")
    plt.tight_layout()
    plt.show()


def plot_coefficients_fs(models, model_names):
    """
    Plots the distribution of coefficients across multiple linear models.
    Works only for linear models trained on X_full_sel.
    """

    feature_names = X_full_sel.columns

    # Build coefficient matrix
    coef_df = pd.DataFrame({
        name: model.coef_
        for model, name in zip(models, model_names)
    }, index=feature_names)

    # Density plot of coefficients
    plt.figure(figsize=(12, 7))
    coef_df.plot(kind="density", figsize=(12, 6))
    plt.title("Distribution of Coefficients Across Linear Models")
    plt.xlabel("Coefficient Value")
    plt.grid(True)
    plt.show()

    # Top absolute coefficients (average magnitude)
    top_abs = coef_df.abs().mean(axis=1).sort_values(ascending=False).head(20)

    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_abs.values, y=top_abs.index, orient="h")
    plt.title("Most Influential Features (Average Across Models)")
    plt.xlabel("Average |coefficient|")
    plt.tight_layout()
    plt.show()


<a id="tree"></a>
## Tree-Based Models Visualizations
(RandomForest, ExtraTrees, HGB)

In [ ]:
def plot_feature_importance(model, model_name, top=20):
    """
    Plots feature importances.
    """

    importances = model.feature_importances_
    feat_names = X_full_final.columns

    imp_df = pd.DataFrame({
        "feature": feat_names,
        "importance": importances
    }).sort_values("importance", ascending=False).head(top)

    plt.figure(figsize=(10, top * 0.35 + 1))
    sns.barplot(x="importance", y="feature", data=imp_df, palette="viridis")
    plt.title(f"{model_name} - Top {top} Feature Importances")
    plt.tight_layout()
    plt.show()


def plot_feature_importance_fs(model, model_name, top=20):
    """
    Plots feature importances for tree-based models.
    Uses the correct FE+FS feature matrix (X_full_sel).
    """

    if not hasattr(model, "feature_importances_"):
        raise ValueError(f"{model_name} does not expose feature_importances_.")

    importances = model.feature_importances_
    feat_names = X_full_sel.columns

    imp_df = pd.DataFrame({
        "feature": feat_names,
        "importance": importances
    }).sort_values("importance", ascending=False).head(top)

    plt.figure(figsize=(10, top * 0.35 + 1))
    sns.barplot(x="importance", y="feature", data=imp_df, palette="viridis")
    plt.title(f"{model_name} - Top {top} Feature Importances")
    plt.tight_layout()
    plt.show()


In [ ]:
def compute_split_frequency(model):
    '''
    Presents how often a feature is used to split. 
    Not as interpretable as features importances, permutable or SHAP, but still good for checking which features drive the tree structure.
    '''

    from collections import Counter
    feature_counts = Counter()

    for tree in model.estimators_:
        tree_ = tree.tree_
        feature = tree_.feature

        for f in feature:
            if f >= 0:  # -2 is "leaf"
                feature_counts[f] += 1

    # Map back to names
    freq_df = pd.DataFrame({
        "feature": [X_full_final.columns[i] for i in feature_counts.keys()],
        "split_count": list(feature_counts.values())
    }).sort_values("split_count", ascending=False)

    return freq_df


def compute_split_frequency_fs(model):
    """
    Counts how often each feature is used to split across all trees.
    Works for tree-based models trained on X_full_sel.
    """

    from collections import Counter
    feature_counts = Counter()

    # Loop through each decision tree inside the ensemble
    for tree in model.estimators_:
        tree_ = tree.tree_
        feature_idx = tree_.feature  # integer indices referring to X_full_sel.columns

        for f in feature_idx:
            if f >= 0:  # -2 indicates a leaf node
                feature_counts[f] += 1

    # Correct feature-name mapping for FE+FS
    feat_names = X_full_sel.columns

    freq_df = pd.DataFrame({
        "feature": [feat_names[i] for i in feature_counts.keys()],
        "split_count": list(feature_counts.values())
    }).sort_values("split_count", ascending=False)

    return freq_df